## Classification Evaluation

In [ ]:
%load_ext autoreload
%autoreload 2

# Run in parent dir
import os

os.chdir("../")

import json
from pathlib import Path

import matplotlib.pyplot as plt
import nako
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score

from models import models
from utils import helpers

### Parameters

In [ ]:
method = 'finetune'
backbone = 'inceptionv3'
img_size= (224, 224)
kfold = 1
experiment_name = f'clf_{backbone}_{img_size[0]}_{method}_{kfold}'

feature_name = 'basis_sex'

device = 'cuda:0'
batch_size = 256
num_workers = 0

# Where results will be saved
project_dir = Path.cwd()
models_dir = project_dir.joinpath('checkpoints')
models_dir.mkdir(parents=True, exist_ok=True)
model_file = models_dir.joinpath(f'{experiment_name}.pt')
results_file = models_dir.joinpath(f'{experiment_name}.json')

plots_dir = project_dir.joinpath('plots')
plots_dir.mkdir(parents=True, exist_ok=True)

### Load results

In [ ]:
with open(results_file, 'r') as f:
       results = json.load(f)

loss_train, loss_val, ids, types, targets, preds, probs = [np.array(results[k]) for k in ['loss_train', 'loss_val', 'ids', 'type', 'targets', 'preds', 'probs']]
del results

### Learning Curve

In [ ]:
fig, ax = plt.subplots(1, 1)
ax.plot(range(loss_train.shape[0]), loss_train, '-o')
ax.plot(range(loss_val.shape[0]), loss_val, '-o')
ax.set_title('Training Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Average Loss')
ax.legend(['Train', 'Val'])
plt.tight_layout()

### Load Dataset

In [ ]:
nako_paths = nako.get_nako_paths(dataset='590', image_res=str(img_size[0]))
transform = models.get_augmentations(img_size, normalization=nako.NORMALIZATION)
dataset_train = nako.NakoDataset(nako_paths, nako.IMAGE_TYPES, transform=transform['train'], split='train', kfold=kfold, feature_name=feature_name, drop_nan=True)
dataset_test = nako.NakoDataset(nako_paths, nako.IMAGE_TYPES, transform=transform['test'], split='test', kfold=kfold, feature_name=feature_name, drop_nan=True)

mapping = dataset_test.mapping
n_classes = len(mapping)
feature_label = dataset_test.feature_label

### Distribution of train and test set

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(14, 5), sharey=True)
ax = ax.flatten()

for i, targets in enumerate([dataset_train.labels, dataset_test.labels]):
    df = pd.DataFrame(data=targets, columns=[feature_label])
    df[feature_label] = df[feature_label].map(lambda x: mapping[x])
    sns.histplot(x=feature_label, data=df, weights=(np.ones(df.shape[0]) / df.shape[0]), multiple='dodge', shrink=0.7, ax=ax[i])
    ax[i].bar_label(ax[i].containers[0], fmt='%.2f')
    ax[i].set_xlabel(feature_label)

ax[0].set_ylabel('Percentage of participants')
ax[0].set_title(f'{feature_label} Distribution Training Set')
ax[1].set_title(f'{feature_label} Distribution Test Set')
plt.tight_layout()

### Accuracy

In [ ]:
acc_test = accuracy_score(targets, preds)
auc_test = roc_auc_score(targets, probs)

print(f'Accuracy = {acc_test:.2f}')
print(f'AUC (test) = {auc_test:.2f}')

### Confusion Matrix

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 5))

cf_matrix = confusion_matrix(targets, preds, normalize='true')
sns.heatmap(cf_matrix, xticklabels=list(mapping.values()), yticklabels=list(mapping.values()), annot=True, fmt='.2f', cmap='rocket_r', annot_kws={"size": 14}, ax=ax)

ax.set_ylabel(f'Real {feature_label}')
ax.set_xlabel(f'Predicted {feature_label}')
ax.set_title('Confusion Matrix Test Set (normalized by row)')
plt.tight_layout()

### Aggregate K-fold results on test set
The classification_val.py does the same on the validation set.

In [ ]:
ex = 'clf_resnet18_224_finetune_'
experiment_names = [ex + str(i) for i in range(1, 6)]

acc, auc = [], []
for experiment_name in experiment_names:
    results_file = models_dir.joinpath(f'{experiment_name}.json')
    with open(results_file, 'r') as f:
        results = json.load(f)

    targets, preds, probs = [np.array(results[k]) for k in ['targets', 'preds', 'probs']]
    del results

    acc_test = accuracy_score(targets, preds)
    auc_test = roc_auc_score(targets, probs)

    acc.append(acc_test)
    auc.append(auc_test)

df_kfold = pd.DataFrame({'Accuracy': acc, 'AUC': auc})
df_kfold

In [ ]:
df_kfold.describe()

### Compute performance estimates
Using bootstraping on the test set.

In [ ]:
experiment_names = ['clf_efficientnetv2_224_finetune_2']

for experiment_name in experiment_names:
    print(experiment_name.split(sep='_')[1])
    results_file = models_dir.joinpath(f'{experiment_name}.json')

    with open(results_file, 'r') as f:
        results = json.load(f)

    targets, preds, probs = [np.array(results[k]) for k in ['targets', 'preds', 'probs']]

    helpers.bootstrap_test(preds, targets, [accuracy_score])
    helpers.bootstrap_test(probs, targets, [roc_auc_score])